# PCA on returns

Same pipeline as `simple_PCA`, but on **daily returns** (simple/percent returns, *not* log) instead of raw prices. Returns are the stationary increments, so PC1 stops being 'the trend' and becomes genuine co-movement structure.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stonks import get_prices, to_returns, load_universe

%matplotlib inline


## Parameters

Same as before: last 2 years, most-liquid `TOP_N` stocks, adjusted close.


In [ ]:
PERIOD = "2y"
INTERVAL = "1d"
TOP_N = 50
FIELD = "close"   # adjusted close -> total returns


## Fetch prices


In [ ]:
prices = get_prices(top_n=TOP_N, period=PERIOD, interval=INTERVAL, field=FIELD)
prices


## Returns (daily, simple)

`to_returns` gives $R_t = P_t/P_{t-1} - 1$ (no log). This is the N×(T−1) matrix of **percent moves** — the stationary increments we actually want to model. (The leading column with no prior day is dropped.)


In [ ]:
returns = to_returns(prices)
print("returns shape:", returns.shape, "| prices shape:", prices.shape)
returns


### What do daily returns look like?

A quick peek at the distribution — this is what we'd eventually put a likelihood on (Gaussian? heavier tails?).


In [ ]:
vals = returns.to_numpy().ravel()
vals = vals[~np.isnan(vals)]
plt.hist(vals, bins=80)
plt.title("Distribution of daily simple returns")
plt.xlabel("daily return"); plt.ylabel("count")


## Clean & centre

Drop stocks with any missing days, then centre each stock's returns by its own mean return.


In [ ]:
returns_clean = returns.dropna()
X = returns_clean.to_numpy(dtype=float)
tickers = list(returns_clean.index)
scale = np.sqrt(len(tickers))   # sqrt(N): unit-variance loadings (~ N(0,1) under noise)

X_centered = X - X.mean(axis=1, keepdims=True)
print("X.shape:", X.shape, "| N stocks:", len(tickers))


## Covariance of returns

Stock×stock covariance of centred returns, normalised by **T** (the time axis). Unlike the price covariance, this measures co-*movement* (percent moves), not level.


In [ ]:
cov = (X_centered @ X_centered.T) / X_centered.shape[1]

plt.imshow(cov)
plt.colorbar(label="covariance")
plt.title("Return covariance"); plt.xlabel("stock"); plt.ylabel("stock")


## PCA via SVD

Compare the variance fractions to `simple_PCA` (where PC1 was ~98%). On returns the spectrum is much flatter — PC1 is now a real co-movement factor, not a trend.


In [ ]:
U, S, Vt = np.linalg.svd(cov)
var = S ** 2 / (S ** 2).sum()
print("variance explained: PC1=%.1f%%  PC2=%.1f%%  PC3=%.1f%%" % (var[0]*100, var[1]*100, var[2]*100))


## Stocks in PCA space

Same `sqrt(N)` Gaussian scaling as before. `(U[i,0], U[i,1])` is stock *i*'s coordinate on PC1/PC2.


In [ ]:
uni = load_universe()
NAMES = dict(zip(uni["ticker"], uni["name"]))

def label(t: str) -> str:
    return f"{t} ({NAMES.get(t, '')[:22]})"


In [ ]:
coords = U[:, :2] * scale
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(coords[:, 0], coords[:, 1], s=25)
for i, t in enumerate(tickers):
    ax.annotate(t, (coords[i, 0], coords[i, 1]), fontsize=7, alpha=0.8)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("Stocks in PCA space (return covariance, top 2 components)")


## Which stocks are similar?

Nearest neighbours in the top-`k` PCA space — now in *co-movement*, not drift.


In [ ]:
k = 5
Z = U[:, :k] * scale
D = np.sqrt(((Z[:, None, :] - Z[None, :, :]) ** 2).sum(-1))
np.fill_diagonal(D, np.inf)

iu = np.triu_indices(len(tickers), k=1)
print("most similar pairs:")
for r in np.argsort(D[iu])[:10]:
    i, j = iu[0][r], iu[1][r]
    print(f"  d={D[i, j]:.3f}  {label(tickers[i]):30s} ~ {label(tickers[j])}")


In [ ]:
target = "AAPL"
i = tickers.index(target)
j = int(D[i].argmin())
print(f"{label(target)}'s nearest neighbour: {label(tickers[j])}  (d={D[i, j]:.3f})")


### Takeaway

On returns PC1 is the **market factor** (everyone moves together ~β=1), not the 2-year trend, and the variance is spread across many components instead of ~98% in one. This return matrix is the right object to model with a likelihood (Gaussian as a baseline; heavier-tailed if the histogram above demands it), and the natural next step is a low-rank **factor model** — the direct analog of PNMF.
